In [0]:
%run ../FASE1/00_utils

In [0]:
%run ./00_utility

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
catalog = get_catalog()
print("Catalog: ", catalog)
STORICO_PROGRAMMI = f"{catalog}.whatif.storico_programmi"
dbutils.widgets.text("STORICO_PROGRAMMI", STORICO_PROGRAMMI)

## Caricamento e modifiche `v_trasmesso_ta`
- Rimozione delle colonne non necessarie e deduplicazione dei record.
- Calcolo della media di `ETA_MEDIA` per ciascun gruppo di colonne.
- Rinomina delle colonne principali: `DATA` in `Data`, `DES_TITOLO_PRINC_INT` in `Programma`, `DES_NETWORK` in `Canale`.
- Conversione degli orari di inizio e fine trasmissione in secondi.
- Pulizia dei valori testuali tramite `trim` e normalizzazione dei nomi dei canali.

In [0]:
df_dettagli = (
    spark.table('ta_prod.silver.v_trasmesso_ta')
    .drop(*COLUMNS_TO_REMOVE_V_TRASMESSO_TA)
    .distinct()
)

df_dettagli = (
    df_dettagli.groupBy([col for col in df_dettagli.columns if col != 'ETA_MEDIA'])
    .agg(F.mean("ETA_MEDIA").alias("ETA_MEDIA"))
)

df_dettagli = df_dettagli.withColumnsRenamed({
    'DATA': 'Data',
    'DES_TITOLO_PRINC_INT': 'Programma',
    'DES_NETWORK': 'Canale'
})

df_dettagli = convert_time_to_seconds(df_dettagli, ['ORA_INIZIO_TRX', 'ORA_FINE_TRX'])

for column in [
    'Programma',
    'Canale',
    'DES_GENERE_ESTESA_INT',
    'DES_GENERE_FILM_INT',
    'DES_GENERE_SPORT_INT',
    'DES_MANIFESTAZIONE_SPORT_INT',
    'DES_SPECIALITA_SPORT_INT'
]:
    df_dettagli = df_dettagli.withColumn(column, F.trim(column))

df_dettagli = df_dettagli.withColumn(
    "Canale",
    F.when(F.col("Canale") == "Italia1", "Italia 1")
     .when(F.col("Canale") == "Canale5", "Canale 5")
     .when(F.col("Canale") == "Dmax Tv", "Dmax")
     .when(F.col("Canale") == "FRISBEE", "Frisbee")
     .when(F.col("Canale") == "Italia 2", "Italia 2 Mediaset")
     .when(F.col("Canale") == "K2 (nazionale)", "K2")
     .when(F.col("Canale") == "Rai GULP", "Rai Gulp")
     .when(F.col("Canale") == "Rai YoYo", "Rai Yoyo")
     .when(F.col("Canale") == "Rete4", "Rete 4")
     .when(F.col("Canale") == "Sky TG24", "Sky Tg24")
     .when(F.col("Canale") == "TV8", "Tv8")
     .otherwise(F.col('Canale'))
)

df_dettagli.limit(100).display()

## Caricamento e modifiche `v_report_programmi`
- Caricamento del dataset e normalizzazione degli orari (`Minuto`, `ORA_INIZIO_TRX`, `ORA_FINE_TRX`) in secondi.
- Rimozione delle colonne e delle righe non necessarie.
- Aggregazione dei dati e pulizia dei record.
- Unione con il dataset dettagliato dei programmi (`df_dettagli`).
- Creazione di una colonna ID univoca per ogni programma.

In [0]:
df_programmi = spark.table('ta_prod.gold.v_report_programmi')

df_programmi = convert_time_to_seconds(
    df_programmi, 
    ['Minuto', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX']
)

df_programmi = df_programmi.drop(
    *set(COLUMNS_TO_REMOVE + FIRST_SREEN_LIVE_VOSDAL_REGIONI + VOD_COLUMNS)
)

df_programmi = df_programmi.dropna(
    subset=['LiveVOSDAL', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX']
)

df_programmi = groupby_single_row(df_programmi)
df_programmi = remove_contenitori(df_programmi)
# df_programmi = compute_total_share(df_programmi)
df_programmi = compute_audience_category_percentage(df_programmi)

# Join con df_dettagli
df_programmi = df_programmi.join(
    df_dettagli, 
    ['Data', 'Canale', 'ORA_INIZIO_TRX', 'ORA_FINE_TRX', 'Programma'], 
    'left'
)

df_programmi = df_programmi.dropna(subset=['Share'])
df_programmi = df_programmi.dropDuplicates(
    ['Data', 'Canale', 'Programma', 'ORA_INIZIO_TRX']
)

df_programmi = df_programmi.withColumn(
    'ID', 
    F.concat(
        F.col('Canale'), F.lit('_'), 
        F.col('Data'), F.lit('_'), 
        F.col('Programma'), F.lit('_'), 
        F.col('ORA_INIZIO_TRX')
    )
)

## Creazione UDF per normalizzazione programmi

In [0]:
# Creiamo delle UDF perché le funzioni di normalizzazione sono definite in Python
normalize_title_udf = F.udf(normalize_title, StringType())
apply_manual_mapping_udf = F.udf(apply_manual_mapping, StringType())

# Creiamo una nuova colonna con i nomi programma normalizzati
df_programmi = df_programmi.withColumn(
    "programma_norm",
    normalize_title_udf(F.col("Programma"))
)
df_programmi = df_programmi.withColumn(
    "programma_norm",
    apply_manual_mapping_udf(F.col("Canale"), F.col("programma_norm"))
)

In [0]:
df_programmi.orderBy('Data', ascending = False).display()

## Salvataggio tabella

In [0]:
# Scrittura dei dati in tabella
df_programmi.write.mode('overwrite').saveAsTable(STORICO_PROGRAMMI)

# Liquid Clustering su Data e Canale
# Forza la raccolta delle statistiche Delta sulle colonne del clustering
spark.sql(f"ALTER TABLE {STORICO_PROGRAMMI} SET TBLPROPERTIES ('delta.dataSkippingStatsColumns' = 'Data,Canale')")
spark.sql(f"ALTER TABLE {STORICO_PROGRAMMI} CLUSTER BY (Data, Canale)")
spark.sql(f"OPTIMIZE {STORICO_PROGRAMMI}")

In [0]:
df_programmi = spark.table(STORICO_PROGRAMMI)
df_programmi.display()

In [0]:
print('Numero righe:  ', df_programmi.count())
print('Numero colonne:', len(df_programmi.columns))